In [ ]:
# --- Load dataset (compact keeps all obs including terminal frames) ---
_, (train_dataset, _) = ogbench.make_env_and_datasets(
    dataset_name=DATASET_NAME,
    dataset_only=True,
    compact_dataset=True,
)

observations = train_dataset["observations"]  # (N, H, W, C)
actions      = train_dataset["actions"]        # (N, 5)
terminals    = train_dataset["terminals"]      # (N,) 1.0 at episode end

# Split flat dataset into list of episodes
terminal_idxs = np.where(terminals == 1.0)[0]
episode_slices = []
start = 0
for end in terminal_idxs:
    episode_slices.append(slice(start, end + 1))
    start = end + 1

print(f"Loaded {len(episode_slices)} episodes, obs shape: {observations.shape}")

In [ ]:
# --- Inspector state ---
saved_segments: list[dict] = []
pending_start: int | None = None

# --- Widgets ---
ep_slider    = widgets.IntSlider(value=0, min=0, max=len(episode_slices)-1, description="Episode", continuous_update=False, layout=widgets.Layout(width="600px"))
frame_slider = widgets.IntSlider(value=0, min=0, max=0, description="Frame",   continuous_update=False, layout=widgets.Layout(width="600px"))
mark_start_btn  = widgets.Button(description="Mark Start",   button_style="info")
mark_end_btn    = widgets.Button(description="Mark End & Add", button_style="success")
clear_btn       = widgets.Button(description="Clear Pending",  button_style="warning")
save_btn        = widgets.Button(description="Save All Segments", button_style="danger")
status_label    = widgets.Label(value="")
segments_output = widgets.Output()
img_output      = widgets.Output()


ep_slider.observe(on_episode_change, names="value")
frame_slider.observe(on_frame_change, names="value")
mark_start_btn.on_click(on_mark_start)
mark_end_btn.on_click(on_mark_end)
clear_btn.on_click(on_clear)
save_btn.on_click(on_save)

# --- Layout ---
controls = widgets.VBox([
    ep_slider,
    frame_slider,
    widgets.HBox([mark_start_btn, mark_end_btn, clear_btn, save_btn]),
    status_label,
])
display(widgets.HBox([widgets.VBox([controls, segments_output]), img_output]))

# Init
on_episode_change(None)
update_segments_view()

NameError: name 'widgets' is not defined

In [ ]:
# --- Load and inspect saved segments ---
with open(SAVE_PATH, "rb") as f:
    loaded = pickle.load(f)

print(f"{len(loaded)} segments loaded:")
for i, seg in enumerate(loaded):
    print(f"  [{i}] episode={seg['episode']}  frames={seg['start']}–{seg['end']}  obs={seg['observations'].shape}  actions={seg['actions'].shape}")